# Stitch Integration (Notebook 7 — course P5)

This notebook follows the **Foreign Whispers P5 stitch** stage from the project handout:

1. **ffmpeg audio remux** — copy the original **video** stream as-is (no re-encode), replace the **audio** track with the dubbed TTS WAV (`POST /api/stitch/{video_id}`).
2. **WebVTT alongside** — rolling two-line **Spanish** captions live as a **separate** file under `pipeline_data/api/dubbed_captions/` (materialized when you call `GET /api/captions/{video_id}` or when the Dubbing Studio loads subtitles). They are **not** burned into the MP4 in the canonical pipeline.

Verify playback **with captions** using the **Dubbing Studio** (`http://localhost:8501`): it attaches the VTT to the `<video>` element. If you only double-click the MP4 in QuickTime, you may see **no** subtitles (expected) and **“incompatible media”** for **AV1** video — that is a player/codec limitation, not a missing stitch step.

**Prerequisites:**
- Docker stack running (`docker compose --profile nvidia up -d`).
- API at `http://localhost:8080`.
- Prior stages (download, transcribe, translate, TTS) complete for the target video.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-stitch")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /Users/raramand/Desktop/Projects/NYU/Year1/Sem2/foreign-whispers
Logfire not configured — using no-op shim.


## Stitch Video (P5 — audio remux)

Call `POST /api/stitch/{video_id}?config=…` to **remux** with ffmpeg: copy the original **video** stream and replace **audio** with the TTS WAV (no video re-encode). This is the handout’s **Stitch** step.

Rolling **WebVTT** is **not** written by this call; it is produced **alongside** when `GET /api/captions/{video_id}` runs (or when the Dubbing Studio loads subtitles).

In [2]:
video_id = "GYQ5yGV_-Oc"  # replace with your video ID

with logfire.span("stitch", video_id=video_id):
    result = fw.stitch(video_id)

print(f"Video ID:   {result['video_id']}")
print(f"Video path: {result['video_path']}")
print(f"Config:     {result['config']}")

Video ID:   GYQ5yGV_-Oc
Video path: /app/pipeline_data/api/dubbed_videos/c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.mp4
Config:     c-fb1074a


## Inspect Output Artifacts

The stitch stage produces files in two directories:

- `pipeline_data/api/dubbed_videos/{config}/` — final dubbed MP4 files
- `pipeline_data/api/dubbed_captions/` — target-language VTT caption files

In [3]:
dubbed_videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_videos"
dubbed_captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_captions"

print("Dubbed videos:")
for f in sorted(dubbed_videos_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(dubbed_videos_dir)}  ({size_mb:.1f} MB)")

print()
print("Dubbed captions:")
for f in sorted(dubbed_captions_dir.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(dubbed_captions_dir)}  ({size_kb:.1f} KB)")

Dubbed videos:
  c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.mp4  (27.7 MB)

Dubbed captions:
  Strait of Hormuz disruption threatens to shake global economy.vtt  (15.7 KB)


## View generated captions (WebVTT alongside)

Rolling two-line **WebVTT** lives under `pipeline_data/api/dubbed_captions/` once the API has materialized it (same rolling format as `GET /api/captions/{video_id}`: current line on top, previous line below).

In [4]:
# Find the VTT file for this video
vtt_files = list(dubbed_captions_dir.rglob(f"{video_id}*.vtt"))
if not vtt_files:
    vtt_files = list(dubbed_captions_dir.rglob("*.vtt"))

if vtt_files:
    vtt_path = vtt_files[0]
    print(f"Caption file: {vtt_path.name}\n")
    lines = vtt_path.read_text().splitlines()
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"\n... ({len(lines) - 30} more lines)")
    print()
    print("Note the rolling two-line pattern: each cue shows the current")
    print("translated line on top and the previous line on the bottom,")
    print("providing continuity for the viewer.")
else:
    print("No VTT files found. Run the stitch step first.")

Caption file: Strait of Hormuz disruption threatens to shake global economy.vtt

WEBVTT

1
00:00:02.320 --> 00:00:09.920
¿Cuál es el peor escenario que te preocupa?

2
00:00:09.920 --> 00:00:16.000
Es que está cerrado durante semanas y semanas y semanas aquí, y usted comienza a ver el mundo
¿Cuál es el peor escenario que te preocupa?

3
00:00:16.000 --> 00:00:21.560
la economía realmente impactada porque es la sangre de la vida en cierta medida.
Es que está cerrado durante semanas y semanas y semanas aquí, y usted comienza a ver el mundo

4
00:00:21.560 --> 00:00:25.880
Así que la realidad es, cuanto más tiempo siga esto, mayor impacto tendrá
la economía realmente impactada porque es la sangre de la vida en cierta medida.

5
00:00:25.880 --> 00:00:27.760
en todas las industrias, en todas las regiones.
Así que la realidad es, cuanto más tiempo siga esto, mayor impacto tendrá

6
00:00:28.048 --> 00:00:33.688
Esta semana en 60 minutos reportamos sobre el Estrecho de Hormuz, esa estrecha v

## Playback (course workflow)

1. **Dubbing Studio:** Open <http://localhost:8501>, complete the pipeline through **Stitch**, then select the **dubbed** variant. The UI loads the MP4 and `/api/captions/...` together (captions visible).

2. **Grading bundle:** Zip the **dubbed MP4** plus the **`.vtt`** from `dubbed_captions/` (per handout P5 outputs). Do not commit large binaries to git.

3. **QuickTime / AV1:** The downloaded YouTube video is often **AV1**; QuickTime may show “incompatible media.” Use **VLC**, the Studio, or optionally `scripts/burn_in_dubbed_preview.py` for a separate **H.264 + hard-sub** preview file (not part of the canonical pipeline).

## Summary (P5)

- **Dubbed MP4** — `pipeline_data/api/dubbed_videos/{config}/<title>.mp4`: ffmpeg **remux** (copy video, replace audio with TTS).
- **WebVTT** — `pipeline_data/api/dubbed_captions/<title>.vtt`: rolling two-line Spanish cues, written when captions are served (`GET /api/captions/{video_id}`) or used by the Studio.

Together these match the course “MP4 + VTT” deliverable; the video stream itself is not re-encoded at stitch time.